# 📰 SCRAPING MEDIA BERITA ONLINE REGIONAL (RISET FISIP - SCOPUS)
**Studi Komparatif Diplomasi Publik Indonesia & Isu Melanesia di ASEAN**

Notebook ini dirancang untuk mengumpulkan korpus artikel berita dari media massa arus utama di **Singapura**, **Malaysia**, **Kawasan Pasifik**, dan **Media Global**.

### 🎯 Tujuan Metodologis (Mengacu Dokumen `MABA SCOPUS.pdf`):
1. **Menjawab RQ1**: Memetakan konstruksi narasi dominan media (*Entman Framing*: HAM, Kedaulatan, Diplomasi, Pembangunan) terkait Papua dan identitas kemelanesiaan.
2. **Menjawab RQ2**: Mengambil korpus berita yang menjadi konsumsi kelompok *attentive public* (kelompok terdidik/pemerhati isu internasional) di Malaysia dan Singapura.
3. **Menjawab RQ3**: Mengukur efektivitas diplomasi publik eksternal Indonesia dalam membingkai narasi kedaulatan di media regional.
4. **Triangulasi Data (Golden Triangle)**: Melengkapi korpus platform **X (Twitter)** dan **YouTube** menjadi kesatuan dataset multi-platform yang utuh.

### 📦 Langkah 1: Instalasi Pustaka Dependensi
Kita menginstall `googlenewsdecoder` untuk mendekode URL Google News RSS secara presisi, serta `beautifulsoup4` dan `tqdm` untuk ekstraksi artikel.

In [ ]:
# Instalasi dependensi
!pip install googlenewsdecoder beautifulsoup4 requests pandas tqdm

### ⚙️ Langkah 2: Konfigurasi Sumber Media & Parameter Sampling
Fokus media berita dipilih secara ketat sesuai kriteria sampling dokumen FISIP:
- **Singapura**: *CNA (Channel NewsAsia)*, *The Straits Times*
- **Malaysia**: *The Star*, *Free Malaysia Today*, *New Straits Times*, *Astro AWANI*, *Malay Mail*
- **Kawasan Pasifik & Melanesia**: *RNZ Pacific*, *ABC News Australia*, *Asia Pacific Report*
- **Mitra Global**: *Al Jazeera English*, *BenarNews*

In [ ]:
import os
import sys
import time
import hashlib
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
from datetime import datetime
import pandas as pd
import requests
from bs4 import BeautifulSoup
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable, *args, **kwargs):
        return iterable
import googlenewsdecoder

# Target Media & Kueri Terfokus
TARGET_NEWS_MEDIA = [
    # 1. SINGAPURA (Blok ASEAN - Menjawab RQ2)
    {
        "country": "Singapore",
        "bloc": "Blok ASEAN",
        "media": "CNA (Channel NewsAsia)",
        "location": "Singapore",
        "queries": [
            "site:channelnewsasia.com \"West Papua\"",
            "site:channelnewsasia.com \"Papua\" Indonesia"
        ]
    },
    {
        "country": "Singapore",
        "bloc": "Blok ASEAN",
        "media": "The Straits Times",
        "location": "Singapore",
        "queries": [
            "site:straitstimes.com \"West Papua\"",
            "site:straitstimes.com \"Papua\" Indonesia"
        ]
    },

    # 2. MALAYSIA (Blok ASEAN - Menjawab RQ2)
    {
        "country": "Malaysia",
        "bloc": "Blok ASEAN",
        "media": "The Star",
        "location": "Kuala Lumpur, Malaysia",
        "queries": [
            "site:thestar.com.my \"West Papua\"",
            "site:thestar.com.my \"Papua\" Indonesia"
        ]
    },
    {
        "country": "Malaysia",
        "bloc": "Blok ASEAN",
        "media": "Free Malaysia Today",
        "location": "Petaling Jaya, Malaysia",
        "queries": [
            "site:freemalaysiatoday.com \"West Papua\"",
            "site:freemalaysiatoday.com \"Papua\" Indonesia"
        ]
    },
    {
        "country": "Malaysia",
        "bloc": "Blok ASEAN",
        "media": "New Straits Times",
        "location": "Kuala Lumpur, Malaysia",
        "queries": [
            "site:nst.com.my \"West Papua\"",
            "site:nst.com.my \"Papua\" Indonesia"
        ]
    },
    {
        "country": "Malaysia",
        "bloc": "Blok ASEAN",
        "media": "Astro AWANI",
        "location": "Kuala Lumpur, Malaysia",
        "queries": [
            "site:astroawani.com \"Papua\"",
            "site:astroawani.com \"West Papua\""
        ]
    },
    {
        "country": "Malaysia",
        "bloc": "Blok ASEAN",
        "media": "Malay Mail",
        "location": "Petaling Jaya, Malaysia",
        "queries": [
            "site:malaymail.com \"West Papua\"",
            "site:malaymail.com \"Papua\" Indonesia"
        ]
    },

    # 3. KAWASAN PASIFIK & MELANESIA (Blok Pasifik & Melanesia)
    {
        "country": "New Zealand",
        "bloc": "Blok Pasifik & Melanesia",
        "media": "RNZ Pacific",
        "location": "Wellington, New Zealand",
        "queries": [
            "site:rnz.co.nz \"West Papua\"",
            "site:rnz.co.nz \"Papua\" Indonesia"
        ]
    },
    {
        "country": "Australia",
        "bloc": "Blok Pasifik & Melanesia",
        "media": "ABC News Australia",
        "location": "Sydney, Australia",
        "queries": [
            "site:abc.net.au \"West Papua\"",
            "site:abc.net.au \"Papua\" Indonesia"
        ]
    },
    {
        "country": "Pacific Regional",
        "bloc": "Blok Pasifik & Melanesia",
        "media": "Asia Pacific Report",
        "location": "Auckland, New Zealand",
        "queries": [
            "site:asiapacificreport.nz \"West Papua\"",
            "site:asiapacificreport.nz \"Melanesia\" Papua"
        ]
    },

    # 4. MITRA GLOBAL (Mitra Global)
    {
        "country": "Pasifik & Global",
        "bloc": "Mitra Global",
        "media": "Al Jazeera English",
        "location": "Doha / Regional Bureau",
        "queries": [
            "site:aljazeera.com \"West Papua\"",
            "site:aljazeera.com \"Papua\" Indonesia"
        ]
    },
    {
        "country": "Regional Asia-Pacific",
        "bloc": "Mitra Global",
        "media": "BenarNews",
        "location": "Washington / Regional Bureau",
        "queries": [
            "site:benarnews.org \"West Papua\"",
            "site:benarnews.org \"Papua\""
        ]
    }
]

MANDATORY_CORE_SUBJECTS = ["papua", "west papua", "papua barat", "melanesia", "melanesian", "ulmwp"]
EXCLUDE_DISASTER_TERMS = ["magnitude", "gempa", "earthquake", "tsunami", "aftershocks", "volcano", "letusan"]

print(f"Konfigurasi siap! {len(TARGET_NEWS_MEDIA)} portal berita terdaftar.")

### 🔍 Langkah 3: Fungsi Ekstraksi & Validasi Akademik
Fungsi ini membersihkan noise HTML, memvalidasi bahwa artikel secara substansial membahas isu diplomasi/kedaulatan/HAM Papua, serta menolak artikel bencana alam tanpa kaitan isu.

In [ ]:
def fetch_rss_items(query, max_items=60):
    encoded_q = urllib.parse.quote(query)
    rss_url = f"https://news.google.com/rss/search?q={encoded_q}&hl=en&gl=US&ceid=US:en"
    req = urllib.request.Request(rss_url, headers={
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    })
    items_data = []
    try:
        res = urllib.request.urlopen(req, timeout=12)
        root = ET.fromstring(res.read())
        for item in root.findall('.//item')[:max_items]:
            title = item.find('title').text if item.find('title') is not None else ""
            link = item.find('link').text if item.find('link') is not None else ""
            pub_date = item.find('pubDate').text if item.find('pubDate') is not None else ""
            if title and link:
                items_data.append({
                    "rss_title": title,
                    "rss_link": link,
                    "pub_date": pub_date
                })
    except Exception as e:
        print(f"      [!] Gagal mengambil RSS: {e}")
    return items_data

def decode_google_news_url(rss_url):
    try:
        res = googlenewsdecoder.new_decoderv1(rss_url)
        if res.get("status"):
            return res.get("decoded_url")
    except Exception:
        pass
    return None

def extract_article_content(url, session):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9,id;q=0.8,ms;q=0.7'
    }
    try:
        resp = session.get(url, headers=headers, timeout=12, allow_redirects=True)
        if resp.status_code != 200:
            return None
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        for noise in soup.find_all(['script', 'style', 'nav', 'header', 'footer', 'aside', 'noscript', 'svg']):
            noise.decompose()
        
        title = ""
        h1 = soup.find('h1')
        if h1 and len(h1.get_text().strip()) > 10:
            title = h1.get_text().strip()
        elif soup.title:
            title = soup.title.get_text().strip()
            
        article_elem = soup.find('article') or soup.find('div', class_=lambda c: c and any(k in c.lower() for k in ['content', 'article', 'body', 'story', 'post-text']))
        search_scope = article_elem if article_elem else soup
        
        paragraphs = []
        for p in search_scope.find_all('p'):
            text = p.get_text().strip()
            if len(text) > 45 and not any(k in text.lower() for k in ['cookie', 'subscribe', 'terms of use', 'all rights reserved', 'advertisement', 'sign up']):
                paragraphs.append(text)
        
        if not paragraphs:
            return None
            
        return {"title": title, "body": " ".join(paragraphs)}
    except Exception:
        return None

def is_valid_academic_sample(title, body):
    combined = f"{title} {body}".lower()
    if not any(sub in combined for sub in MANDATORY_CORE_SUBJECTS):
        return False
    if any(dis in title.lower() for dis in EXCLUDE_DISASTER_TERMS):
        diplomatic_triggers = ["conflict", "rebel", "military", "tni", "diplomacy", "sovereignty", "rights", "aid", "australia", "indonesia"]
        if not any(dip in combined for dip in diplomatic_triggers):
            return False
    if len(body) < 150:
        return False
    return True

### 🚀 Langkah 4: Jalankan Crawler & Scraper Berita
Eksekusi pengumpulan artikel. Setiap artikel akan diparsing dan distandarisasi ke skema multi-platform.

In [ ]:
output_csv = "news_papua_asean_articles.csv"
max_articles_per_media = 50

session = requests.Session()
collected_articles = []
seen_urls = set()

print("Memulai pengambilan data berita regional...")

for media_cfg in TARGET_NEWS_MEDIA:
    media_name = media_cfg["media"]
    country = media_cfg["country"]
    bloc = media_cfg["bloc"]
    location = media_cfg["location"]
    queries = media_cfg["queries"]
    
    print(f"\n[+] Memproses: {media_name} ({country} | {bloc})")
    media_count = 0
    
    for q in queries:
        if media_count >= max_articles_per_media:
            break
            
        rss_items = fetch_rss_items(q, max_items=40)
        
        for item in tqdm(rss_items, desc=f"    {media_name[:15]}", leave=False):
            if media_count >= max_articles_per_media:
                break
                
            rss_link = item["rss_link"]
            real_url = decode_google_news_url(rss_link)
            
            if not real_url or real_url in seen_urls:
                continue
            seen_urls.add(real_url)
            
            article = extract_article_content(real_url, session)
            if not article:
                continue
                
            title = article["title"] if article["title"] else item["rss_title"]
            body = article["body"]
            
            if not is_valid_academic_sample(title, body):
                continue
            
            raw_date = item.get("pub_date", "")
            try:
                dt = datetime.strptime(raw_date[:25].strip(), "%a, %d %b %Y %H:%M:%S")
                formatted_date = dt.strftime("%Y-%m-%d")
            except Exception:
                formatted_date = datetime.now().strftime("%Y-%m-%d")
            
            article_id = "news_" + hashlib.md5(real_url.encode('utf-8')).hexdigest()[:12]
            full_text = f"{title}. {body}"
            
            collected_articles.append({
                "platform": "News Media",
                "post_id": article_id,
                "created_at": formatted_date,
                "username": media_name + " Desk",
                "user_location": location,
                "country_bloc": bloc,
                "media_source": media_name,
                "full_text": full_text,
                "like_count": 0,
                "url": real_url
            })
            media_count += 1
            time.sleep(0.4)
            
    print(f"    [✓] Didapat {media_count} artikel dari {media_name}")

df = pd.DataFrame(collected_articles)
df.to_csv(output_csv, index=False, encoding='utf-8')
print(f"\n[SELESAI] Total {len(df)} artikel berhasil dikumpulkan ke '{output_csv}'.")

### 📊 Langkah 5: Evaluasi Distribusi Korpus Media Berita

In [ ]:
print("=" * 60)
print("DISTRIBUSI ARTIKEL BERITA PER BLOK GEOPOLITIK:")
print("=" * 60)
print(df["country_bloc"].value_counts())

print("\n" + "=" * 60)
print("DISTRIBUSI ARTIKEL BERITA PER MEDIA OUTLET:")
print("=" * 60)
print(df["media_source"].value_counts())

print("\nContoh 3 Artikel Teratas:")
display(df[["media_source", "country_bloc", "created_at", "full_text"]].head(3))

### 📥 Langkah 6: Download Hasil Dataset Berita

In [ ]:
from google.colab import files
files.download(output_csv)